# 01 · Attention and a GPT from scratch
Scaled dot-product attention written out, checked against PyTorch's fused kernel, then the pretrained character model generates text.

In [1]:
import sys, json
sys.path.insert(0, '..')
import torch
from gaf.common import device
dev = device(); print(torch.__version__, dev)

2.15.0 mps


In [2]:
import torch.nn.functional as F
from gaf.transformer import attention
torch.manual_seed(0)
q, k, v = (torch.randn(1, 2, 6, 4) for _ in range(3))
ours = attention(q, k, v)
print('max |ours - torch SDPA| =', (ours - F.scaled_dot_product_attention(q, k, v, is_causal=True)).abs().max().item())

max |ours - torch SDPA| = 1.4901161193847656e-07


In [3]:
# the causal mask: each position attends only to itself and the past
import math
s = (q @ k.transpose(-2, -1) / math.sqrt(4))[0, 0]
mask = torch.ones(6, 6, dtype=torch.bool).tril()
print(s.masked_fill(~mask, float('-inf')).softmax(-1).round(decimals=2))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3900, 0.6100, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.4000, 0.4000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.1400, 0.3100, 0.1600, 0.0000, 0.0000],
        [0.0600, 0.0800, 0.2200, 0.5600, 0.0800, 0.0000],
        [0.0300, 0.0700, 0.7400, 0.0600, 0.0900, 0.0100]])


In [4]:
from pathlib import Path
from gaf.transformer import GPT, GPTConfig
from experiments.train_gpt import TOK
ck = Path('../checkpoints/gpt_pretrain.pt')
if ck.exists():
    state = torch.load(ck, map_location='cpu')
    m = GPT(GPTConfig(**state['cfg'])); m.load_state_dict(state['model']); m.eval()
    torch.manual_seed(3)
    print(TOK.decode(m.generate(torch.tensor([TOK.encode('## Context\n\n')]), 300)[0].tolist()))
else:
    print('run experiments/train_gpt.py first')

## Context

A ll on the table a snoic on of the earned, and new hrough its in deduplicates
  the queries not numbers. A compaction and is a run a recompled month.

- **The Airflow run most a holdout metaday watermark runs a rows a on mofer is for it can of a lab aron laptop.** The
the lab registry watermark and


In [5]:
r = json.load(open('../results/gpt_pretrain.json'))
print({k: r[k] for k in ('params', 'seconds', 'tokens_per_s', 'uniform_loss', 'best_val', 'best_step')})

{'params': 824064, 'seconds': 47.6, 'tokens_per_s': 257929, 'uniform_loss': 4.7185, 'best_val': 1.5159, 'best_step': 1500}
